## Prerequisites

To execute this tutorial you will need:
* Python 3.10+
* AWS credentials
* Amazon Bedrock AgentCore SDK
* Strands Agents

In [1]:
!pip install --force-reinstall -U -r requirements.txt --quiet

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
autogluon-multimodal 1.5.0 requires nvidia-ml-py3<8.0,>=7.352.0, which is not installed.
autogluon-timeseries 1.5.0 requires chronos-forecasting<2.4,>=2.2.2, which is not installed.
autogluon-timeseries 1.5.0 requires einops<1,>=0.7, which is not installed.
autogluon-timeseries 1.5.0 requires peft<0.18,>=0.13.0, which is not installed.
dash 2.18.1 requires dash-core-components==2.0.0, which is not installed.
dash 2.18.1 requires dash-html-components==2.0.0, which is not installed.
dash 2.18.1 requires dash-table==5.0.0, which is not installed.
jupyter-ai 2.31.7 requires faiss-cpu!=1.8.0.post0,<2.0.0,>=1.8.0, which is not installed.
aiobotocore 2.22.0 requires botocore<1.37.4,>=1.37.2, but you have botocore 1.42.70 which is incompatible.
amazon-sagemaker-sql-editor 0.1.21 requires rpds-py==0.27.1, but you have rpds

In [12]:
# Import all necessary libraries for the tutorial
from bedrock_agentcore_starter_toolkit import Runtime 
from bedrock_agentcore_starter_toolkit.operations.runtime import destroy_bedrock_agentcore
from pathlib import Path
from strands import Agent,tool 
from strands.models import BedrockModel  
from strands.tools.mcp.mcp_client import MCPClient 
from streamable_http_sigv4 import streamablehttp_client_with_sigv4  
from strands_tools import agent_graph, retrieve
import boto3  
import getpass 
import json
import logging  
import uuid 
import os


logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s - %(name)s - %(levelname)s - %(message)s'
)
logger = logging.getLogger(__name__)

In [3]:
session = boto3.Session()

credentials = session.get_credentials()

region = session.region_name

cf_client = boto3.client(
    "cloudformation", region_name=region
)

agentcore_client = boto3.client(
    "bedrock-agentcore-control",
    region_name=region,
)

identity_client = boto3.client(
    "bedrock-agentcore-control",
    region_name=region,
)

s3_client = session.client('s3')

sts_client = session.client('sts')
account_id = sts_client.get_caller_identity()["Account"]

2026-03-18 08:33:59,538 - botocore.credentials - INFO - Found credentials in shared credentials file: ~/.aws/credentials
2026-03-18 08:33:59,544 - botocore.credentials - INFO - Found credentials in shared credentials file: ~/.aws/credentials


In [4]:
# Define configuration variables for the tutorial resources

# Gateway and target names
#gateway_name = "aws-host-support-knowledge-mcp-gatewat"  # Name for the AgentCore Gateway

# Agent configuration
agent_name = "AWS_Support_knowledge_QA_Agent0318"  # Name for the deployed agent in AgentCore Runtime

## (Optional) Step 6: Deploy to Amazon Bedrock AgentCore Runtime

Let's start with our Strands Agent using Amazon Bedrock model. All the others will work exactly the same.

### Step 6.1: Configure Amazon Bedrock AgentCore Runtime

First we will use our starter toolkit to configure the AgentCore Runtime deployment with an entrypoint, the execution role we just created and a requirements file. We will also configure the starter kit to auto create the Amazon ECR repository on launch.

During the configure step, your docker file will be generated based on your application code

<center>

![prereq](./images/configure.png)

</center>

In [28]:
# Initialize the AgentCore Runtime manager
# This object handles the deployment of our agent to AWS Lambda
agentcore_runtime = Runtime()

# Configure the runtime deployment settings
# This prepares all necessary AWS resources for deploying the agent
response = agentcore_runtime.configure(
    entrypoint="aws_support_agent.py",  
    auto_create_ecr=True, 
    requirements_file="requirements.txt",  
    region=region,  
    agent_name=agent_name,
    auto_create_execution_role=True,
    non_interactive=True
)

# Display the configuration response
response

Entrypoint parsed: file=/home/sagemaker-user/aws-omni-support-agent/04_create_knowledge_mcp_gateway_Agent/aws_support_agent.py, bedrock_agentcore_name=aws_support_agent
2026-03-18 09:17:04,178 - bedrock_agentcore_starter_toolkit.utils.runtime.entrypoint - INFO - Entrypoint parsed: file=/home/sagemaker-user/aws-omni-support-agent/04_create_knowledge_mcp_gateway_Agent/aws_support_agent.py, bedrock_agentcore_name=aws_support_agent
Memory disabled - agent will be stateless
2026-03-18 09:17:04,178 - bedrock_agentcore_starter_toolkit.notebook.runtime.bedrock_agentcore - INFO - Memory disabled - agent will be stateless
Configuring BedrockAgentCore agent: AWS_Support_knowledge_QA_Agent0318
2026-03-18 09:17:04,179 - bedrock_agentcore_starter_toolkit.operations.runtime.configure - INFO - Configuring BedrockAgentCore agent: AWS_Support_knowledge_QA_Agent0318


💡 No container engine found (Docker/Finch/Podman not installed)

✓ Default deployment uses CodeBuild (no container engine needed), For local builds, install Docker, Finch, or 
Podman

Memory disabled
2026-03-18 09:17:04,223 - bedrock_agentcore_starter_toolkit.operations.runtime.configure - INFO - Memory disabled
Network mode: PUBLIC
2026-03-18 09:17:04,226 - bedrock_agentcore_starter_toolkit.operations.runtime.configure - INFO - Network mode: PUBLIC


⚠️ Platform mismatch: Current system is 'linux/amd64' but Bedrock AgentCore requires 'linux/arm64', so local builds
won't work.
Please use default launch command which will do a remote cross-platform build using code build.For deployment other
options and workarounds, see: 
https://docs.aws.amazon.com/bedrock-agentcore/latest/devguide/getting-started-custom.html

📄 Generated Dockerfile: 
/home/sagemaker-user/aws-omni-support-agent/04_create_knowledge_mcp_gateway_Agent/Dockerfile

Generated .dockerignore: /home/sagemaker-user/aws-omni-support-agent/04_create_knowledge_mcp_gateway_Agent/.dockerignore
2026-03-18 09:17:04,232 - bedrock_agentcore_starter_toolkit.operations.runtime.configure - INFO - Generated .dockerignore: /home/sagemaker-user/aws-omni-support-agent/04_create_knowledge_mcp_gateway_Agent/.dockerignore
Keeping 'AWS_Support_knowledge_QA_Agent0318' as default agent
2026-03-18 09:17:04,238 - bedrock_agentcore_starter_toolkit.utils.runtime.config - INFO - Keeping 'AWS_Support_knowledge_QA_Agent0318' as default agent
Bedrock AgentCore configured: /home/sagemaker-user/aws-omni-support-agent/04_create_knowledge_mcp_gateway_Agent/.bedrock_agentcore.yaml
2026-03-18 09:17:04,240 - bedrock_agentcore_starter_toolkit.notebook.runtime.bedrock_agentcore - INFO - Bedrock AgentCore configured: /home/sagemaker-user/aws-omni-support-agent/04_create_knowledge_mcp_gateway_Agent/.bedrock_agentcore.yaml


ConfigureResult(config_path=PosixPath('/home/sagemaker-user/aws-omni-support-agent/04_create_knowledge_mcp_gateway_Agent/.bedrock_agentcore.yaml'), dockerfile_path=PosixPath('/home/sagemaker-user/aws-omni-support-agent/04_create_knowledge_mcp_gateway_Agent/Dockerfile'), dockerignore_path=PosixPath('/home/sagemaker-user/aws-omni-support-agent/04_create_knowledge_mcp_gateway_Agent/.dockerignore'), runtime='None', runtime_type=None, region='us-east-1', account_id='887221633712', execution_role=None, ecr_repository=None, auto_create_ecr=True, s3_path=None, auto_create_s3=False, memory_id=None, network_mode='PUBLIC', network_subnets=None, network_security_groups=None, network_vpc_id=None)

### Step 6.2: Launch Amazon Bedrock AgentCore Runtime

Now that we've got a docker file, let's launch the agent to the AgentCore Runtime. This will create the Amazon ECR repository and the AgentCore Runtime

<center>

![prereq](./images/launch.png)

</center>

In [29]:
msg = "Launching Agent to AgentCore Runtime (This might take several minutes...)"
print(msg)
launch_result = agentcore_runtime.launch(auto_update_on_conflict=True)
print("✅ Launch completed")

🚀 Launching Bedrock AgentCore (cloud mode - RECOMMENDED)...
2026-03-18 09:17:09,753 - bedrock_agentcore_starter_toolkit.notebook.runtime.bedrock_agentcore - INFO - 🚀 Launching Bedrock AgentCore (cloud mode - RECOMMENDED)...
   • Deploy Python code directly to runtime
2026-03-18 09:17:09,754 - bedrock_agentcore_starter_toolkit.notebook.runtime.bedrock_agentcore - INFO -    • Deploy Python code directly to runtime
   • No Docker required (DEFAULT behavior)
2026-03-18 09:17:09,755 - bedrock_agentcore_starter_toolkit.notebook.runtime.bedrock_agentcore - INFO -    • No Docker required (DEFAULT behavior)
   • Production-ready deployment
2026-03-18 09:17:09,755 - bedrock_agentcore_starter_toolkit.notebook.runtime.bedrock_agentcore - INFO -    • Production-ready deployment

2026-03-18 09:17:09,755 - bedrock_agentcore_starter_toolkit.notebook.runtime.bedrock_agentcore - INFO - 
💡 Deployment options:
2026-03-18 09:17:09,756 - bedrock_agentcore_starter_toolkit.notebook.runtime.bedrock_agentcore -

Launching Agent to AgentCore Runtime (This might take several minutes...)
✅ Reusing existing ECR repository: 887221633712.dkr.ecr.us-east-1.amazonaws.com/bedrock-agentcore-aws_support_knowledge_qa_agent0318


Getting or creating CodeBuild execution role for agent: AWS_Support_knowledge_QA_Agent0318
2026-03-18 09:17:10,010 - bedrock_agentcore_starter_toolkit.services.codebuild - INFO - Getting or creating CodeBuild execution role for agent: AWS_Support_knowledge_QA_Agent0318
Role name: AmazonBedrockAgentCoreSDKCodeBuild-us-east-1-6818eec47e
2026-03-18 09:17:10,011 - bedrock_agentcore_starter_toolkit.services.codebuild - INFO - Role name: AmazonBedrockAgentCoreSDKCodeBuild-us-east-1-6818eec47e
Reusing existing CodeBuild execution role: arn:aws:iam::887221633712:role/AmazonBedrockAgentCoreSDKCodeBuild-us-east-1-6818eec47e
2026-03-18 09:17:10,068 - bedrock_agentcore_starter_toolkit.services.codebuild - INFO - Reusing existing CodeBuild execution role: arn:aws:iam::887221633712:role/AmazonBedrockAgentCoreSDKCodeBuild-us-east-1-6818eec47e
Using dockerignore.template with 47 patterns for zip filtering
2026-03-18 09:17:10,106 - bedrock_agentcore_starter_toolkit.services.codebuild - INFO - Using doc

✅ Launch completed


In [30]:
print(f"Agent ARN: {launch_result.agent_arn}")
print(f"Agent ID: {launch_result.agent_id}")

os.environ['AGENT_ARN'] = launch_result.agent_arn

Agent ARN: arn:aws:bedrock-agentcore:us-east-1:887221633712:runtime/AWS_Support_knowledge_QA_Agent0318-gmVJN24PZ1
Agent ID: AWS_Support_knowledge_QA_Agent0318-gmVJN24PZ1


### Step 6.3: Invoke AgentCore runtime

Finally, we can invoke our AgentCore Runtime with a payload

<center>

![prereq](./images/invoke.png)

</center>

In [31]:
import time

print("Checking AgentCore Runtime status...")
status_response = agentcore_runtime.status()
status = status_response.endpoint['status']
print(f"Initial status: {status}")

end_status = ['READY', 'CREATE_FAILED', 'DELETE_FAILED', 'UPDATE_FAILED']
max_wait_time = 1800  # 30 minutes
wait_time = 0

while status not in end_status and wait_time < max_wait_time:
    msg = f"Status: {status} - waiting... ({wait_time}s/{max_wait_time}s)"
    print(msg)
    time.sleep(10)
    wait_time += 10

    status_response = agentcore_runtime.status()
    status = status_response.endpoint['status']

if wait_time >= max_wait_time:
    print(f"⚠️ Timeout reached. Current status: {status}")
elif status == 'READY':
    print(f"✅ AgentCore Runtime is READY!")
else:
    print(f"⚠️ AgentCore Runtime status: {status}")

print(f"Final status: {status}")

Retrieved Bedrock AgentCore status for: AWS_Support_knowledge_QA_Agent0318
2026-03-18 09:18:21,717 - bedrock_agentcore_starter_toolkit.notebook.runtime.bedrock_agentcore - INFO - Retrieved Bedrock AgentCore status for: AWS_Support_knowledge_QA_Agent0318


Checking AgentCore Runtime status...
Initial status: READY
✅ AgentCore Runtime is READY!
Final status: READY


## 4.Grant premission

In [ ]:
grant "ssm:GetParameter"/"bedrock:Retrieve"/"bedrock-agentcore:InvokeGateway" to this agentcore runtime 

In [33]:
from agent_client import test_invoke_agent

# Test the deployed support case agent\n
test_invoke_agent()

Testing with prompt: 帮我查看一下case id:177376641300818的内容，总结给我
Using agent ARN: arn:aws:bedrock-agentcore:us-east-1:887221633712:runtime/AWS_Support_knowledge_QA_Agent0318-gmVJN24PZ1
Response status: 200
Content type: text/event-stream; charset=utf-8
Agent response:
## 📋 工单内容总结

**工单基本信息：**
| 项目 | 内容 |
|------|------|
| 工单ID | 177376641300818 |
| 主题 | 如何清空一个S3存储桶 |
| 状态 | ✅ 已解决 (resolved) |
| 服务 | Amazon S3 (amazon-simple-storage-service) |
| 类别 | 一般指导 (general-guidance) |
| 严重程度 | Low (低) |
| 创建时间 | 2026-03-17 16:53:33 (UTC) |
| 提交者 | havpan@amazon.com |

---

### 📝 问题描述

用户询问如何清空Amazon S3存储桶（删除桶内所有对象但保留存储桶本身），具体包括：
1. 通过AWS控制台如何操作
2. 通过AWS CLI如何操作
3. 启用版本控制的存储桶如何处理
4. 大型存储桶的推荐方法

---

### 💡 AWS技术支持回复要点

AWS提供了全面的解决方案：

**1. 控制台方法：**
- **通用存储桶**：S3控制台 → 选择存储桶 → 清空 → 输入\permanently delete\"确认
- **目录存储桶**：类似步骤，在目录存储桶导航中操作

**2. AWS CLI方法：**
- **非版本控制存储桶**：
  ```bash
  aws s3 rm s3://bucket-name --recursive
  ```

**3. 版本控制存储桶处理：**
- 需要分别删除所有对象版本和删除标记：
  ```bash
  # 删除版本控制对象
  aws s3api dele

True

# Congratulations!